# ML-05 stretch — Feature Vector and Leakage/Privacy Check (optional)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuan05-afk/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

Core of this card now lives in ML-04; this notebook is the explicit leakage hunt.


## 1. Build the feature vector


In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

ROOT = Path('.').resolve()
# Support running from repo root or work/notebooks
if (ROOT / 'data/raw/content_refresh_anonymized.csv').exists():
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
elif (ROOT.parents[1] / 'data/raw/content_refresh_anonymized.csv').exists():
    ROOT = ROOT.parents[1]
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
else:
    # Colab / nested
    for cand in [ROOT, *ROOT.parents]:
        if (cand / 'data/raw/content_refresh_anonymized.csv').exists():
            ROOT = cand
            DATA = cand / 'data/raw/content_refresh_anonymized.csv'
            break
    else:
        raise FileNotFoundError('data/raw/content_refresh_anonymized.csv')

WORK_OUT = ROOT / 'work' / 'outputs'
WORK_OUT.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA)
print(f'Loaded {len(df):,} rows × {df.shape[1]} cols from {DATA.relative_to(ROOT)}')
print(f'Clients (pseudonymous): {df["client_id"].nunique()}')

sys.path.insert(0, str(ROOT / 'scripts'))
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES
print('Numeric features:', len(MODEL_NUMERIC_FEATURES))
print('Categorical features:', len(MODEL_CATEGORICAL_FEATURES))
print('Leakage candidates excluded from MODEL_* lists: trend_direction, trend_pct')


Loaded 30,000 rows × 44 cols from data\raw\content_refresh_anonymized.csv
Clients (pseudonymous): 32
Numeric features: 18
Categorical features: 8
Leakage candidates excluded from MODEL_* lists: trend_direction, trend_pct


## 2. Feature notes (meaning, missing, categorical, available-when?)


In [2]:
notes = pd.DataFrame({
    'feature': ['ctr','avg_position','days_since_last_update','word_count','ai_traffic_pct'],
    'note': [
        '×100 percent; not a 0-1 fraction',
        '0 = missing rank, not position zero',
        'freshness; safe feature',
        'missingness follows content_type',
        'can exceed 100; cross-system rate',
    ]
})
print(notes.to_string(index=False))


               feature                                note
                   ctr    ×100 percent; not a 0-1 fraction
          avg_position 0 = missing rank, not position zero
days_since_last_update             freshness; safe feature
            word_count    missingness follows content_type
        ai_traffic_pct   can exceed 100; cross-system rate


## 3. The leakage hunt


In [3]:
forbidden = {'trend_direction','trend_pct','is_declining_label','content_id','client_id'}
sys.path.insert(0, str(ROOT / 'scripts'))
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES
used = set(MODEL_NUMERIC_FEATURES) | set(MODEL_CATEGORICAL_FEATURES)
leak = used & forbidden
print('Forbidden fields in feature lists?', leak or 'NONE — clean')
# Correlation trap: trend_pct vs label if someone cheated
df['is_declining_label'] = (df['trend_direction'].str.lower()=='down').astype(int)
if 'trend_pct' in df.columns:
    print('trend_pct corr with label (must NOT be a feature):',
          round(df['trend_pct'].corr(df['is_declining_label']), 3))


Forbidden fields in feature lists? NONE — clean
trend_pct corr with label (must NOT be a feature): -0.141


## 4. What I excluded and why

Excluded `trend_direction` / `trend_pct` (define the label), IDs (privacy + non-generalizing),
and any text/URL fields (not present; would be identifying).


In [4]:
print('Excluded: trend_direction, trend_pct, content_id, client_id as features.')
print('Privacy: no client names/URLs/queries in this notebook.')


Excluded: trend_direction, trend_pct, content_id, client_id as features.
Privacy: no client names/URLs/queries in this notebook.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
